In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
from crewai import Agent, Task, Crew, Process
from crewai_tools import FileReadTool
import pandas as pd

In [ ]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())  # .env 파일에서 OPENAI_API_KEY 로드

# .env 파일에 OPENAI_API_KEY=sk-... 형태로 저장 후 사용
# (Never hardcode API keys — add .env to .gitignore)
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o'

### Agent 페르소나 정의

In [ ]:
# 툴 설정: 재고 파일을 읽기 위한 도구
inventory_tool = FileReadTool(file_path='./amore_inventory.csv')

In [ ]:
# 분석 전문가: 날씨와 민감도를 고려한 성분 가이드라인 제시
skin_analyst = Agent(
    role='개인 맞춤형 피부 과학 및 성분 분석가',
    goal='사용자의 다양한 피부 상태와 환경적 요인을 종합 분석하여 최적의 케어 방향 설정',
    backstory="""당신은 수만 명의 피부 데이터를 다뤄본 전문가입니다. 
    사용자의 입력에서 [피부 타입, 현재 고민, 민감도, 사용 목적]을 스스로 추출합니다.
    계절이나 외부 환경(날씨)이 피부에 미치는 영향을 고려하여 전문가적인 진단을 내립니다.
    '노화' 대신 '생기 케어'나 '시간을 되돌리는 케어'와 같은 긍정적인 단어를 사용합니다.
    전문 용어(예: 상피세포 성장인자, 경피 수분 손실 등)는 일반인이 이해하기 쉬운 말로 풀어서 설명합니다.""",
    verbose=False,
    allow_delegation=False
)

# 재고 매칭 전문가: 실제 amore_inventory.csv에서 제품 검색
inventory_manager = Agent(
    role='재고 최적화 및 제품 매칭 전문가',
    goal='amore_inventory.csv 데이터를 조회하고 분석된 피부 진단 결과에 가장 부합하는 실제 제품을 재고 데이터에서 선정',
    backstory="""당신은 아모레퍼시픽의 실시간 재고 관리자입니다.
    당신은 amore_inventory.csv 파일 데이터를 반영해서 피부 분석 결과가 제안하는 성분과 기능을 가진 제품 중, 재고가 충분한 제품을 우선 매핑합니다.""",
    verbose=False,
    tools=[inventory_tool],
    allow_delegation=False
)

# 마케팅 전문가 : 마케팅 문구 생성
marketing_specialist = Agent(
    role='뷰티 콘텐츠 마케터',
    goal='따뜻하고 이해하기 쉬운 문체로 복잡한 분석 결과를 고객이 이해하기 쉬운 매력적인 추천 보고서로 변환',
    backstory="""당신은 고객의 요청에 공감하며 전문적인 해결책을 제시하는 카운셀러입니다.
    고객의 언어로 친절하게 답변하되, 필수 정보는 정확히 전달합니다
    당신은 잡지 에디터처럼 세련되면서도 친절한 문체를 사용합니다. 
    고객이 자신의 피부 상태에 대해 기분 나쁘지 않도록 배려 섞인 조언을 건넵니다.""",
    verbose=False,
    allow_delegation=False
)

### Task 정의

In [ ]:
# Task 1: 고객 데이터 분석
task_analysis = Task(
    description="""고객의 요청[{user_request}]을 정밀 분석하세요.
    1. 사용자의 주된 피부 고민과 피부 타입을 식별하세요.
    2. 현재 요청과 관련된 외부 요인(날씨, 계절 등)이 피부에 미치는 영향을 추론하세요.
    3. 해당 상태를 개선하기 위해 필요한 핵심 성분과 피해야 할 성분 가이드라인을 작성하세요.""",
    agent=skin_analyst,
    expected_output="종합 피부 진단 결과 및 맞춤형 성분 가이드라인"
)

# Task 2: 제품 및 재고 매핑
task_mapping = Task(
    description="""1. 앞선 진단 결과를 바탕으로 amore_inventory.csv 파일에서 가장 적합한 제품을 검색하세요.
    2. 제품의 '핵심성분'과 '효과'가 고객의 고민 해결에 적절한지 대조하세요.
    3. 재고 상황(충분 > 보통 > 부족 순)을 고려하여 최종 1개의 제품을 선정하세요.
    4. 에이전트 간의 대화 과정이나 조회한 CSV 데이터 리스트 전체를 결과값에 절대 포함하지 마세요.
    5. 오직 '선정된 제품의 정보'만 다음 에이전트에게 전달하세요.""",
    agent=inventory_manager,
    expected_output="데이터 기반 최종 추천 제품 정보"
)

# Task 3: 최종 추천 문장 생성 (Response 구현)
task_report = Task(
    description="""고객을 위한 최종 추천 리포트를 작성하세요.
    1. 이전 에이전트들이 분석한 '전체 제품 리스트'나 '중간 분석 과정' csv 파일에 있는 내용은 최종 리포트에 절대 포함하지 마세요. 
    2. 오직 선정된 '하나의 제품'에 대한 결과만 출력하세요.
    3. 모든 내용은 한국어로 작성하세요.

    출력 형식:
    ✨ 아모레퍼시픽 AI 카운셀러의 응답:
    - 제품명: [선정된 제품명]
    - 고객 피부 타입 및 적합 성분: [에이전트가 분석한 고객의 타입을 배려 있게 설명하고, 이해하기 쉬운 성분 이야기 포함]
    - 해당 제품의 효과: [제품 사용 후 좋아질 모습 설명]""",
    
    agent=marketing_specialist,
    expected_output="마크다운 형식의 개인화된 고객 맞춤형 마케팅 문구 및 데이터 태그(한국어)"
)

### Crew 설정 및 실행

In [ ]:
beauty_crew = Crew(
    agents=[skin_analyst, inventory_manager, marketing_specialist],
    tasks=[task_analysis, task_mapping, task_report],
    process=Process.sequential # 순차적으로 작업 수행
)

# 실시간 요청 처리 함수 (CSV 로드 대신 직접 호출)
def get_realtime_recommendation(user_input):
    """
    사용자의 자연어 요청을 직접 받아 AI 에이전트 팀을 가동합니다.
    """
    print(f"\n--- 사용자 요청 분석 시작: {user_input} ---")
    
    # 에이전트 팀 가동
    result = beauty_crew.kickoff(inputs={
        "user_request": user_input
    })
    
    if isinstance(result, str):
        return result
    return result.raw

### 결과출력

In [ ]:
# 실제 실행 예시
user_request_example1 = "나는 건성에 성분에 민감한 타입이고 요즘 피부가 푸석해"
response1 = get_realtime_recommendation(user_request_example1)

print("\n" + "="*50)
print(response1)

In [ ]:
# 실제 실행 예시 2
user_request_example2 = "나는 민감성에 홍조가 있고 카모마일 향을 선호해"
response2 = get_realtime_recommendation(user_request_example2)

print("\n" + "="*50)
print(response2)

In [ ]:
# 실제 실행 예시 3
user_request_example3 = "나는 복합성에 유분과 모공이 고민이야"
response3 = get_realtime_recommendation(user_request_example3)

print("\n" + "="*50)
print(response3)